# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id` as per the Croissant data model.

## Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- Citation: Liu, Y., Duan, X., Yang, S., Zhang, Y. and Han, S. 2026. Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution. Frontiers.

In [ ]:
# Install mlcroissant (if not installed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review all available record sets, fields, and their `@id` values for referencing in further steps.

In [ ]:
# List record sets (each identified by @id)
record_sets = [rs for rs in getattr(metadata, 'record_set', [])]
if not record_sets:
    # Try the alternative (for Croissant 1.x): in the JSON it's `recordSet`
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # Fall back: Try to infer from records API in case none are under metadata
    print("No record sets found in metadata! Scanning for available record sets from the dataset...")
    schema_record_sets = [r['@id'] for r in dataset.records_schema()]
    print(f"Available record set @ids from Croissant schema: {schema_record_sets}")
else:
    print("Available record sets and their @id values:")
    for record_set in record_sets:
        rec_set_id = getattr(record_set, '@id', str(record_set))
        print("-", rec_set_id)
        # List the fields
        if hasattr(record_set, 'field'):
            fields = record_set.field
        elif hasattr(record_set, 'fields'):
            fields = record_set.fields
        else:
            fields = []
        print("  fields:")
        for field in fields:
            field_id = getattr(field, '@id', str(field))
            print(f"    - {field_id}")

# As `mlcroissant` exposes dataset.records_schema(), let's show all recordset @ids with their fields:
print("\nRecord sets from Croissant schema (via mlcroissant.records_schema()):")
for rec_schema in dataset.records_schema():
    rec_id = rec_schema['@id']
    print(f"- Record set @id: {rec_id}")
    if 'field' in rec_schema:
        print("  Fields:")
        for field in rec_schema['field']:
            print(f"    - {field['@id']}")

## 3. Data Extraction
Load data for all record sets into pandas DataFrames, referencing each by its `@id`.

You can edit the record set list or field list below to select specific subsets for further analysis.

In [ ]:
# Discover all available record sets from the schema
record_sets = [rec['@id'] for rec in dataset.records_schema()]
print("Extracting the following record sets by @id:", record_sets)

# Load each record set and show its columns
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded record set @id: {record_set_id}")
    print("Columns (@id):", list(df.columns))

# Display the head of the first available record set
if record_sets:
    first_rs = record_sets[0]
    print(f"\nFirst records from record set @id: {first_rs}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering, normalization, and grouping using fields by their `@id`.

We'll demonstrate the following steps on one of the numeric fields, referencing it by its `@id`. Adjust the variable `sample_numeric_field_id` to the numeric field you wish to analyze.

In [ ]:
# --- EDA for a numeric field --- #
# Use your preferred record set and field @ids from the overview above.

# Select the record set and field to analyze by their @id
selected_record_set_id = record_sets[0]  # Change this if you want to analyze a different record set
df = dataframes[selected_record_set_id]

# Find all numeric field columns by inspecting the DataFrame
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields found (@id): {numeric_cols}")

# If there are numeric columns available, pick the first for demonstration
if numeric_cols:
    sample_numeric_field_id = numeric_cols[0]  # Replace with your selected field if desired
    # Basic descriptive stats
    print(df[sample_numeric_field_id].describe())
    
    threshold = df[sample_numeric_field_id].mean()
    filtered_df = df[df[sample_numeric_field_id] > threshold]
    print(f"\nFiltered records where {sample_numeric_field_id} > {threshold}")
    print(filtered_df[[sample_numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{sample_numeric_field_id}_normalized"] = (
        (filtered_df[sample_numeric_field_id] - filtered_df[sample_numeric_field_id].mean()) /
        filtered_df[sample_numeric_field_id].std()
    )
    print(f"\nNormalized {sample_numeric_field_id}:\n", filtered_df[[sample_numeric_field_id, f"{sample_numeric_field_id}_normalized"]].head())
    
    # Optionally group by a categorical field (if present)
    # List candidate group columns
    candidate_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print("Candidate grouping fields (@id):", candidate_group_fields)
    if candidate_group_fields:
        group_field_id = candidate_group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[sample_numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {sample_numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found in this record set.")

## 5. Visualization
Visualize distributions and relationships between fields using `matplotlib` or `seaborn` as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For demonstration, plot a histogram of a numeric field, referencing by its @id
if numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[sample_numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {sample_numeric_field_id}")
    plt.xlabel(sample_numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Optionally, scatter against a grouping field if possible
if numeric_cols and candidate_group_fields:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field_id, y=sample_numeric_field_id)
    plt.title(f"{sample_numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing the FAIR^2 colorectal cancer survivor dataset via the Croissant schema using `mlcroissant`.

**Key observations:**
- Dataset structure and content are accessible programmatically via `@id` for each entity (record set, field, column).
- Columns and field IDs can be dynamically queried and referenced.
- For more advanced analysis, refer to the schema's official documentation and experiment with domain-specific features.

**Remember:** Always reference record sets, fields, and columns in this dataset by their `@id` for full reproducibility.